# Titanic
Suited for binary logistic regression

In [ ]:
# Necessary libraries for data manipulation and visualization
import kagglehub
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Necessary libraries of scikit-learn for machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Download latest version
path = kagglehub.dataset_download("heptapod/titanic")

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv('/run/media/rifat/Felicitous/Machine_Learning_Specialization/Logistic_Regression/datasets/train_and_test2.csv')
df.sample(5)

### Exploratory Data Analysis

In [ ]:
print(df.info())

In [ ]:
# Check for columns with only one unique value
zero_cols = [c for c in df.columns if df[c].nunique() == 1]
cols_to_drop = zero_cols + [c for c in df.columns if c == 'Passengerid']

df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
print(df)

In [ ]:
df.rename(columns={'2urvived': 'Survived', 'sibsp': 'siblings_spouses', 'Parch': 'parents_children', 'Pclass': 'Passenger_Class'}, inplace=True)
df.info()

In [ ]:
for c in df.columns:
    print(f"Column: {c}, Unique Values: {df[c].nunique()}")

In [ ]:
correlation = df.corr()['Survived'].sort_values(ascending=False)
print(correlation)

In [ ]:
# Correlation visualization using matplotlib
plt.style.use('dark_background')

plt.figure(figsize=(10, 8))
corr_matrix = df.corr()
plt.imshow(corr_matrix, cmap='seismic', aspect='auto')

plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha='right')
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.colorbar(label='Correlation')
plt.title('Correlation with Survived')
plt.xlabel('Features')
plt.ylabel('Features')
plt.show()

### Train Test Split

In [ ]:

X = df.drop('Survived', axis=1)
y = df['Survived']

In [ ]:
# Check for missing values in the features or null
print(X.isnull().sum(),'\n')
print((y.value_counts()))

In [ ]:
print(X['Embarked'].fillna(X['Embarked'].mode()[0], inplace=True).isnull().sum())

In [ ]:
# Fill missing values in the 'Embarked' column with the mode (most frequent value)
X['Embarked'] = X['Embarked'].fillna(X['Embarked'].mode()[0])
X.loc[X['Embarked'].isnull(), 'Embarked']
print(X['Embarked'].sample(5))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [ ]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


### Feature Scaling

In [ ]:
# Feature scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Training the Logistic Regression model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

In [ ]:
# Coefficients and intercept of the logistic regression model
print(f"Coefficients: \n{model.coef_}\n\nIntercept: {model.intercept_}")

### Prediction

In [335]:
# Predicting the target variable for the test set
y_pred = model.predict(X_test_scaled)
print(f"Predicted values: {y_pred}")

Predicted values: [0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0
 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1 0 0 1 0 0 0 0 0 0 0 1 0 1 0 1 0 0 0 0 0 0 0
 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 1 0 0 0 0
 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1 0 1 0 0 1 0 0 0 0 0 0 0 0 1 0 1
 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 1
 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0
 0 1 1]


In [339]:
# Cross validation 
scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
print(f"Fold scores: {scores}")
print(f"Mean accuracy: {scores.mean():.4f}")
print(f"Std deviation: {scores.std():.4f}")

Fold scores: [0.78571429 0.76190476 0.77990431 0.76555024 0.81339713]
Mean accuracy: 0.7813
Std deviation: 0.0183


In [340]:
# Evaluate the model's performance using accuracy, confusion matrix, and classification report
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}\n")
print(f"Classification Report:\n{classification_report(y_test, y_pred)}\n")

Accuracy: 0.7672

Confusion Matrix:
[[174  15]
 [ 46  27]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.92      0.85       189
           1       0.64      0.37      0.47        73

    accuracy                           0.77       262
   macro avg       0.72      0.65      0.66       262
weighted avg       0.75      0.77      0.74       262


